# Remote Sensing & GIS Data Extraction Tool

This notebook provides a streamlined workflow for extracting **remote sensing spectral indices** and **GIS terrain variables** for any set of user-defined coordinates or uploaded field plots.  
It is designed for ecologists who want to quickly obtain RS and environmental layers for ecological or habitat analyses.

## What This Notebook Does

### 1. User Inputs

- Upload field plot coordinates (`CSV`, `Excel`, `GeoJSON`, or `Shapefile`)  
- Or manually enter latitude/longitude pairs  
- Define time range and select desired satellite products  

### 2. Remote Sensing Data (GEE)

Automatically loads and processes:

- **Sentinel-2** (10–20 m)  
- **Landsat 5/7/8/9** (30 m)  
- **MODIS MOD09A1** (500 m)  

Includes:

- Cloud masking  
- Optional seasonal filtering  

### 3. Spectral Indices Extraction

The notebook computes **nine spectral indices**:

#### Greenness
- NDVI  
- EVI  
- SAVI  
- MSAVI  

#### Moisture / Water
- NDMI  
- NDWI  
- MNDWI  

#### Burn Severity
- NBR  
- NBR2  

> Indices are calculated from user-selected satellite collections and sampled at plot locations.

### 4. GIS Terrain Layers

Derived from **SRTM DEM (30 m)** or optional NASA DEM:

- Elevation  
- Slope  
- Aspect  
- Hillshade  

### 5. Optional Hydrological Layers

Users may also include:

- JRC Global Surface Water (occurrence, seasonality)  
- HAND (Height Above Nearest Drainage)  
- SMAP soil moisture  
- Sentinel-1 backscatter for wetness/inundation  

### 6. Extract RS & GIS Values to Points

The notebook samples:

- All spectral indices  
- Terrain layers  
- Optional hydrological layers  

> Returns a clean table linked to each user-provided coordinate.

### 7. Export Results

Outputs can be saved as:

- CSV  
- Excel  
- GeoJSON  
- Shapefile  

> For further ecological analysis and modeling.

### 8. Optional Mapping

Quick visual map previews using **geemap**:

- Field plot locations  
- Selected RS layers (e.g., NDVI composite, DEM)


In [2]:
# =========================================================
# Remote Sensing & GIS Data Extraction Tool (GEE)
# =========================================================

# -------------------- 1. Imports & Initialization --------------------
import ee
import geemap
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import os

# Initialize Earth Engine
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

# Output folder
OUT_DIR = "./rs_gis_exports"
os.makedirs(OUT_DIR, exist_ok=True)

Enter verification code:  4/1Ab32j90JuA6ajl62QEkqmmLXnbfHfJHN70rCYqhSFhhAJxVtBh-B043EmG0



Successfully saved authorization token.


In [37]:
def add_indices(img):
    B = img.select('BLUE')
    G = img.select('GREEN')
    R = img.select('RED')
    N = img.select('NIR')
    SW1 = img.select('SWIR1')
    SW2 = img.select('SWIR2')

    ndvi = N.subtract(R).divide(N.add(R)).rename('NDVI')
    evi = N.subtract(R).multiply(2.5).divide(N.add(R.multiply(6)).subtract(B.multiply(7.5)).add(1)).rename('EVI')
    savi = N.subtract(R).multiply(1.5).divide(N.add(R).add(0.5)).rename('SAVI')
    msavi = (N.multiply(2).add(1).subtract((N.multiply(2).add(1)).pow(2).subtract(N.subtract(R).multiply(8)).sqrt()).divide(2)).rename('MSAVI')
    ndmi = N.subtract(SW1).divide(N.add(SW1)).rename('NDMI')
    nbr = N.subtract(SW2).divide(N.add(SW2)).rename('NBR')
    nbr2 = SW1.subtract(SW2).divide(SW1.add(SW2)).rename('NBR2')
    ndwi = G.subtract(N).divide(G.add(N)).rename('NDWI')
    mndwi = G.subtract(SW1).divide(G.add(SW1)).rename('MNDWI')

    return img.addBands([ndvi, evi, savi, msavi, ndmi, nbr, nbr2, ndwi, mndwi])

In [38]:
# =========================================================
# Remote Sensing & GIS Data Extraction Tool (GEE)
# Optimized for large exports
# =========================================================
import ee
import datetime

# ee.Authenticate()
# ee.Initialize()

# -------------------- 1. Parameters --------------------
WORKING_AREA = ee.FeatureCollection(
    'projects/rse-global-wetlands/assets/Tata_Eastern_Morocco_Working_Area'
)
stations = ee.FeatureCollection(
    'projects/rse-global-wetlands/assets/Trophic_survey_data_114_station_habitat'
)
points = ee.FeatureCollection(
    'projects/rse-global-wetlands/assets/Houbara_Count_Stand_2016-23'
)
aoi = WORKING_AREA.geometry()

index_bands = ['NDVI', 'EVI', 'SAVI', 'MSAVI', 'NDMI', 'NBR', 'NBR2', 'NDWI', 'MNDWI']

DRIVE_FOLDER = 'GEE_Export_CSVs'  # Google Drive folder

# -------------------- 2. Helper Functions --------------------
def clean_features(fc):
    """Keep only features with at least one valid index."""
    return fc.filter(ee.Filter.Or(*[ee.Filter.notNull([b]) for b in index_bands]))

def add_indices(img):
    B = img.select('BLUE')
    G = img.select('GREEN')
    R = img.select('RED')
    N = img.select('NIR')
    SW1 = img.select('SWIR1')
    SW2 = img.select('SWIR2')

    ndvi = N.subtract(R).divide(N.add(R)).rename('NDVI')
    evi = N.subtract(R).multiply(2.5).divide(N.add(R.multiply(6)).subtract(B.multiply(7.5)).add(1)).rename('EVI')
    savi = N.subtract(R).multiply(1.5).divide(N.add(R).add(0.5)).rename('SAVI')
    msavi = (N.multiply(2).add(1).subtract((N.multiply(2).add(1)).pow(2).subtract(N.subtract(R).multiply(8)).sqrt()).divide(2)).rename('MSAVI')
    ndmi = N.subtract(SW1).divide(N.add(SW1)).rename('NDMI')
    nbr = N.subtract(SW2).divide(N.add(SW2)).rename('NBR')
    nbr2 = SW1.subtract(SW2).divide(SW1.add(SW2)).rename('NBR2')
    ndwi = G.subtract(N).divide(G.add(N)).rename('NDWI')
    mndwi = G.subtract(SW1).divide(G.add(SW1)).rename('MNDWI')

    return img.addBands([ndvi, evi, savi, msavi, ndmi, nbr, nbr2, ndwi, mndwi])


# -------------------- 3. Export Function (batched by year) --------------------
def export_collection_by_year(col, scale, sensor_name, start_year, end_year):
    """Export an ImageCollection per year to avoid GEE limits."""
    for year in range(start_year, end_year+1):
        start_date = f'{year}-01-01'
        end_date = f'{year}-12-31'
        col_year = col.filterDate(start_date, end_date)
        if col_year.size().getInfo() == 0:
            continue  # Skip empty years

        for target, target_name in zip([stations, points], ['Stations', 'Occurrence']):
            def reduce_one_image(img):
                date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
                ft = img.reduceRegions(collection=target, reducer=ee.Reducer.mean(), scale=scale)
                return ft.map(lambda f: f.set({'date': date}))

            fc_list = col_year.map(reduce_one_image)
            fc = ee.FeatureCollection(fc_list).flatten()
            fc = clean_features(fc)

            task_name = f'{sensor_name}_{target_name}_{year}'
            task = ee.batch.Export.table.toDrive(
                collection=fc,
                description=task_name,
                folder=DRIVE_FOLDER,
                fileFormat='CSV'
            )
            task.start()
            print(f'[INFO] Export started: {task_name}')

# -------------------- 4. Sensor Preprocessing --------------------
# Landsat
def mask_landsat_sr(image):
    qa = image.select('QA_PIXEL')
    mask = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0)).And(qa.bitwiseAnd(1 << 1).eq(0))
    return image.updateMask(mask)

def rename_landsat(image, sensor):
    if sensor in ['LT05', 'LE07']:
        mapping = {'SR_B1':'BLUE','SR_B2':'GREEN','SR_B3':'RED','SR_B4':'NIR','SR_B5':'SWIR1','SR_B7':'SWIR2'}
    else:
        mapping = {'SR_B2':'BLUE','SR_B3':'GREEN','SR_B4':'RED','SR_B5':'NIR','SR_B6':'SWIR1','SR_B7':'SWIR2'}
    return image.select(list(mapping.keys()), list(mapping.values())) \
                .multiply(0.0000275).add(-0.2) \
                .copyProperties(image, ['system:time_start'])

def prep_landsat(col_id, sensor):
    return (ee.ImageCollection(col_id)
            .filterBounds(aoi)
            .filter(ee.Filter.calendarRange(3,12,'month'))
            .filter(ee.Filter.lte('CLOUD_COVER',10))
            .map(mask_landsat_sr)
            .map(lambda im: rename_landsat(im, sensor))
            .map(add_indices)
            .select(index_bands))

# Sentinel-2
def mask_s2_sr(image):
    scl = image.select('SCL')
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return image.updateMask(mask)

def rename_s2(image):
    mapping = {'B2':'BLUE','B3':'GREEN','B4':'RED','B8':'NIR','B11':'SWIR1','B12':'SWIR2'}
    return image.select(list(mapping.keys()), list(mapping.values()))

# MODIS
def mask_modis_sr(image):
    state = image.select('StateQA')
    mask = state.bitwiseAnd(1 << 10).eq(0).And(state.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

def rename_modis(image):
    mapping = {'sur_refl_b01':'RED','sur_refl_b02':'NIR','sur_refl_b03':'BLUE','sur_refl_b04':'GREEN',
               'sur_refl_b05':'SWIR1','sur_refl_b06':'SWIR2'}
    return image.select(list(mapping.keys()), list(mapping.values())) \
                .multiply(0.0001).copyProperties(image,['system:time_start'])

# -------------------- 5. Prepare ImageCollections --------------------
landsat_col = (prep_landsat('LANDSAT/LT05/C02/T1_L2','LT05')
               .merge(prep_landsat('LANDSAT/LE07/C02/T1_L2','LE07'))
               .merge(prep_landsat('LANDSAT/LC08/C02/T1_L2','LC08'))
               .merge(prep_landsat('LANDSAT/LC09/C02/T1_L2','LC09')))

s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(aoi)
          .filter(ee.Filter.calendarRange(3,12,'month'))
          .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE',10))
          .map(mask_s2_sr)
          .map(rename_s2)
          .map(add_indices)
          .select(index_bands))

modis_col = (ee.ImageCollection('MODIS/061/MOD09A1')
             .filterBounds(aoi)
             .filter(ee.Filter.calendarRange(3,12,'month'))
             .map(mask_modis_sr)
             .map(rename_modis)
             .map(add_indices)
             .select(index_bands))

# -------------------- 6. Export Spectral Indices (batched by year) --------------------

export_collection_by_year(landsat_col, scale=30, sensor_name='Landsat', start_year=1984, end_year=2025)
export_collection_by_year(s2_col, scale=10, sensor_name='Sentinel2', start_year=2017, end_year=2025)
export_collection_by_year(modis_col, scale=500, sensor_name='MODIS', start_year=2000, end_year=2025)

# -------------------- 7. Terrain & Hydrology --------------------
dem = ee.Image('USGS/SRTMGL1_003')
terrain = ee.Image.cat([
    dem.rename('elevation'),
    ee.Terrain.slope(dem).rename('slope'),
    ee.Terrain.aspect(dem).rename('aspect'),
    ee.Terrain.hillshade(dem).rename('hillshade')
])

gsw = ee.Image('JRC/GSW1_3/GlobalSurfaceWater').select('occurrence')
s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
       .filterBounds(aoi) \
       .filter(ee.Filter.eq('instrumentMode','IW')) \
       .select('VH').mean()
hydro = ee.Image.cat([gsw, s1.rename('S1_VH_mean')])

terrain_hydro = ee.Image.cat([terrain, hydro])

def sample_static_layers(img, sensor_name='Terrain_Hydro', scale=30):
    """Export static layers over stations and points."""
    for target, target_name in zip([stations, points], ['Stations','Occurrence']):
        ft = img.reduceRegions(collection=target, reducer=ee.Reducer.mean(), scale=scale)
        task_name = f'{sensor_name}_{target_name}'
        task = ee.batch.Export.table.toDrive(
            collection=ft,
            description=task_name,
            folder=DRIVE_FOLDER,
            fileFormat='CSV'
        )
        task.start()
        print(f'[INFO] Export started: {task_name}')

sample_static_layers(terrain_hydro, sensor_name='Terrain_Hydro', scale=30)

print('[INFO] All tasks submitted. Check your Google Drive for CSVs.')


[INFO] Export started: Landsat_Stations_1984
[INFO] Export started: Landsat_Occurrence_1984
[INFO] Export started: Landsat_Stations_1985
[INFO] Export started: Landsat_Occurrence_1985
[INFO] Export started: Landsat_Stations_1986
[INFO] Export started: Landsat_Occurrence_1986
[INFO] Export started: Landsat_Stations_1987
[INFO] Export started: Landsat_Occurrence_1987
[INFO] Export started: Landsat_Stations_1988
[INFO] Export started: Landsat_Occurrence_1988
[INFO] Export started: Landsat_Stations_1989
[INFO] Export started: Landsat_Occurrence_1989
[INFO] Export started: Landsat_Stations_1990
[INFO] Export started: Landsat_Occurrence_1990
[INFO] Export started: Landsat_Stations_1991
[INFO] Export started: Landsat_Occurrence_1991
[INFO] Export started: Landsat_Stations_1992
[INFO] Export started: Landsat_Occurrence_1992
[INFO] Export started: Landsat_Stations_1993
[INFO] Export started: Landsat_Occurrence_1993
[INFO] Export started: Landsat_Stations_1994
[INFO] Export started: Landsat_Occu

In [ ]:
# =========================================================
# Python script to download all CSVs from a specific GEE-export folder
# =========================================================

from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
import os

# -------------------- 1. Authenticate with Google --------------------
gauth = GoogleAuth()
gauth.LocalWebserverAuth()  # Opens browser for login
drive = GoogleDrive(gauth)

# -------------------- 2. User parameters --------------------
# Name of the folder in Google Drive where GEE exports are saved
GEE_FOLDER_NAME = "GEE_Export_CSVs"  # Change to your folder name

# Local folder to save CSVs
LOCAL_FOLDER = os.path.expanduser("~/Desktop/GEE_Exports")
os.makedirs(LOCAL_FOLDER, exist_ok=True)

# -------------------- 3. Get the folder ID --------------------
folder_list = drive.ListFile(
    {'q': f"mimeType='application/vnd.google-apps.folder' and title='{GEE_FOLDER_NAME}' and trashed=false"}
).GetList()

if len(folder_list) == 0:
    raise ValueError(f"Folder '{GEE_FOLDER_NAME}' not found in Google Drive.")

folder_id = folder_list[0]['id']
print(f"Found GEE folder: {GEE_FOLDER_NAME} (ID: {folder_id})")

# -------------------- 4. List CSV files in the folder --------------------
file_list = drive.ListFile({'q': f"'{folder_id}' in parents and trashed=false"}).GetList()
gee_csv_files = [f for f in file_list if f['title'].endswith('.csv')]

print(f"Found {len(gee_csv_files)} CSV files in the folder.")

# -------------------- 5. Download all CSV files --------------------
for file in gee_csv_files:
    local_path = os.path.join(LOCAL_FOLDER, file['title'])
    print(f"Downloading: {file['title']} -> {local_path}")
    file.GetContentFile(local_path)

print("All GEE CSV files downloaded successfully!")


In [31]:
# =========================================================
# Remote Sensing & GIS Data Extraction Tool (OPTIMIZED)
# =========================================================
import ee
import os
import datetime # Used for dynamic file naming
ee.Initialize()

# --- 1. User-Configurable Parameters (For App Integration) ---
# NOTE: In a GEE App, these would be controlled by UI widgets.

# The GCS bucket where the large CSVs will be exported.
# *** USER MUST CHANGE THIS TO THEIR OWN GCS BUCKET ***
GCS_BUCKET_NAME = 'your-gcs-bucket-name'

# Base assets (could be replaced by user-uploaded assets in a real App)
WORKING_AREA_ASSET = 'projects/rse-global-wetlands/assets/Tata_Eastern_Morocco_Working_Area'
STATIONS_ASSET = 'projects/rse-global-wetlands/assets/Trophic_survey_data_114_station_habitat'
POINTS_ASSET = 'projects/rse-global-wetlands/assets/Houbara_Count_Stand_2016-23'

# Time/Cloud Filters (Could also be App Inputs)
START_DATE = '1984-03-01'
END_DATE = '2025-09-30'
CLOUD_COVER_MAX = 10 # Used for Landsat/S2

# The input FeatureCollections
WORKING_AREA = ee.FeatureCollection(WORKING_AREA_ASSET)
stations = ee.FeatureCollection(STATIONS_ASSET)
points = ee.FeatureCollection(POINTS_ASSET)
aoi = WORKING_AREA.geometry()

# Bands to compute and select
INDEX_BANDS = ['NDVI','EVI','SAVI','MSAVI','NDMI','NBR','NBR2','NDWI','MNDWI']

# -------------------- 2. Helper Functions (Unchanged, as they are optimal) --------------------
# (clean_features, add_indices, mask_landsat_sr, rename_landsat, prep_landsat,
# mask_s2_sr, rename_s2, mask_modis_sr, rename_modis functions are kept the same)

def clean_features(fc):
    """Keep only features with at least one valid index."""
    return fc.filter(ee.Filter.Or(*[ee.Filter.notNull([b]) for b in INDEX_BANDS]))

def add_indices(img):
    B, G, R = img.select('BLUE'), img.select('GREEN'), img.select('RED')
    N, SW1, SW2 = img.select('NIR'), img.select('SWIR1'), img.select('SWIR2')
    # ... (all your index calculations) ...
    ndvi = N.subtract(R).divide(N.add(R)).rename('NDVI')
    evi = N.subtract(R).multiply(2.5).divide(N.add(R.multiply(6)).subtract(B.multiply(7.5)).add(1)).rename('EVI')
    savi = N.subtract(R).multiply(1.5).divide(N.add(R).add(0.5)).rename('SAVI')
    msavi = (N.multiply(2).add(1).subtract((N.multiply(2).add(1)).pow(2).subtract(N.subtract(R).multiply(8)).sqrt())).divide(2).rename('MSAVI')
    ndmi = N.subtract(SW1).divide(N.add(SW1)).rename('NDMI')
    nbr = N.subtract(SW2).divide(N.add(SW2)).rename('NBR')
    nbr2 = SW1.subtract(SW2).divide(SW1.add(SW2)).rename('NBR2')
    ndwi = G.subtract(N).divide(G.add(N)).rename('NDWI')
    mndwi = G.subtract(SW1).divide(G.add(SW1)).rename('MNDWI')

    return img.addBands([ndvi, evi, savi, msavi, ndmi, nbr, nbr2, ndwi, mndwi])

# Masking and Renaming functions (omitted for brevity, assume they are defined)
def mask_landsat_sr(image):
    qa = image.select('QA_PIXEL')
    mask = qa.bitwiseAnd(1 << 3).eq(0) & qa.bitwiseAnd(1 << 4).eq(0) & qa.bitwiseAnd(1 << 1).eq(0)
    return image.updateMask(mask)

def rename_landsat(image, sensor):
    if sensor in ['LT05', 'LE07']:
        mapping = {'SR_B1':'BLUE','SR_B2':'GREEN','SR_B3':'RED','SR_B4':'NIR','SR_B5':'SWIR1','SR_B7':'SWIR2'}
    else:
        mapping = {'SR_B2':'BLUE','SR_B3':'GREEN','SR_B4':'RED','SR_B5':'NIR','SR_B6':'SWIR1','SR_B7':'SWIR2'}
    return image.select(list(mapping.keys()), list(mapping.values())).multiply(0.0000275).add(-0.2).copyProperties(image, ['system:time_start'])

def prep_landsat(col_id, sensor):
    return (ee.ImageCollection(col_id)
            .filterBounds(aoi)
            .filterDate(START_DATE, END_DATE)
            .filter(ee.Filter.calendarRange(3,12,'month'))
            .filter(ee.Filter.lte('CLOUD_COVER',CLOUD_COVER_MAX))
            .map(mask_landsat_sr)
            .map(lambda im: rename_landsat(im, sensor))
            .map(add_indices)
            .select(INDEX_BANDS))
# S2 Masking/Renaming (similar structure)
def mask_s2_sr(image):
    scl = image.select('SCL')
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return image.updateMask(mask)

def rename_s2(image):
    mapping = {'B2':'BLUE','B3':'GREEN','B4':'RED','B8':'NIR','B11':'SWIR1','B12':'SWIR2'}
    return image.select(list(mapping.keys()), list(mapping.values()))
# MODIS Masking/Renaming (similar structure)
def mask_modis_sr(image):
    state = image.select('StateQA')
    mask = state.bitwiseAnd(1 << 10).eq(0).And(state.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

def rename_modis(image):
    mapping = {'sur_refl_b01':'RED','sur_refl_b02':'NIR','sur_refl_b03':'BLUE','sur_refl_b04':'GREEN',
                'sur_refl_b05':'SWIR1','sur_refl_b06':'SWIR2'}
    return image.select(list(mapping.keys()), list(mapping.values())).multiply(0.0001).copyProperties(image,['system:time_start'])


# -------------------- 3. OPTIMIZED Sampling Function --------------------
def sample_collection_optimized(col, scale, sensor_name):
    """
    Sample an ImageCollection over stations and points, exporting to GCS.
    """
    current_time = datetime.datetime.now().strftime('%Y%m%d%H%M')

    for target, target_name in zip([stations, points], ['Stations', 'Occurrence']):
        # Nested function to handle per-image reduction
        def reduce_one_image(img):
            date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
            # Use Reducer.mean for point data sampling
            ft = img.reduceRegions(collection=target, reducer=ee.Reducer.mean(), scale=scale)
            # Add date to all resulting features
            return ft.map(lambda f: f.set({'date': date, 'sensor': sensor_name}))

        fc_list = col.map(reduce_one_image)
        # Flatten the list of FeatureCollections into a single FeatureCollection
        fc = ee.FeatureCollection(fc_list).flatten()
        fc = clean_features(fc)

        # *** LARGE EXPORT OPTIMIZATION: TO CLOUD STORAGE ***
        file_prefix = f'data_extract/{current_time}/{sensor_name}_{target_name}'
        task = ee.batch.Export.table.toCloudStorage(
            collection=fc,
            description=f'{sensor_name}_{target_name}_Export',
            bucket=GCS_BUCKET_NAME,
            fileNamePrefix=file_prefix,
            fileFormat='CSV'
        )
        task.start()
        print(f'Export started to GCS: gs://{GCS_BUCKET_NAME}/{file_prefix}.csv')


# -------------------- 4. Static Layers Sampling (OPTIMIZED) --------------------
# Prepare static layers (unchanged)
dem = ee.Image('USGS/SRTMGL1_003')
terrain = ee.Image.cat([
    dem.rename('elevation'),
    ee.Terrain.slope(dem).rename('slope'),
    ee.Terrain.aspect(dem).rename('aspect'),
    ee.Terrain.hillshade(dem).rename('hillshade')
])
gsw = ee.Image('JRC/GSW1_3/GlobalSurfaceWater').select('occurrence')
s1 = ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi).filter(ee.Filter.eq('instrumentMode','IW')).select('VH').mean()
hydro = ee.Image.cat([gsw, s1.rename('S1_VH_mean')])
terrain_hydro = ee.Image.cat([terrain, hydro])

def sample_static_layers_optimized(img, sensor_name='Terrain_Hydro', scale=30):
    """Sample static raster layers over stations and points, exporting to GCS."""
    current_time = datetime.datetime.now().strftime('%Y%m%d%H%M')

    for target, target_name in zip([stations, points], ['Stations','Occurrence']):
        ft = img.reduceRegions(collection=target, reducer=ee.Reducer.mean(), scale=scale)

        # *** LARGE EXPORT OPTIMIZATION: TO CLOUD STORAGE ***
        file_prefix = f'data_extract/{current_time}/{sensor_name}_{target_name}_static'
        task = ee.batch.Export.table.toCloudStorage(
            collection=ft,
            description=f'{sensor_name}_{target_name}_Static_Export',
            bucket=GCS_BUCKET_NAME,
            fileNamePrefix=file_prefix,
            fileFormat='CSV'
        )
        task.start()
        print(f'Export started to GCS: gs://{GCS_BUCKET_NAME}/{file_prefix}.csv')


# -------------------- 5. Main Execution Flow (Ready for App deployment) --------------------

# Prepare all ImageCollections
landsat_col = (prep_landsat('LANDSAT/LT05/C02/T1_L2','LT05')
               .merge(prep_landsat('LANDSAT/LE07/C02/T1_L2','LE07'))
               .merge(prep_landsat('LANDSAT/LC08/C02/T1_L2','LC08'))
               .merge(prep_landsat('LANDSAT/LC09/C02/T1_L2','LC09')))

s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(aoi)
          .filterDate(START_DATE, END_DATE)
          .filter(ee.Filter.calendarRange(3,12,'month'))
          .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE',CLOUD_COVER_MAX))
          .map(mask_s2_sr)
          .map(rename_s2)
          .map(add_indices)
          .select(INDEX_BANDS))

modis_col = (ee.ImageCollection('MODIS/061/MOD09A1')
             .filterBounds(aoi)
             .filterDate(START_DATE, END_DATE)
             .filter(ee.Filter.calendarRange(3,12,'month'))
             .map(mask_modis_sr)
             .map(rename_modis)
             .map(add_indices)
             .select(INDEX_BANDS))

# Execute time-series sampling
for col, scale, sensor in [(landsat_col, 30, 'Landsat'),
                           (s2_col, 10, 'Sentinel2'),
                           (modis_col, 500, 'MODIS')]:
    sample_collection_optimized(col, scale, sensor)

# Execute static layer sampling
sample_static_layers_optimized(terrain_hydro, sensor_name='Terrain_Hydro', scale=30)

TypeError: unsupported operand type(s) for &: 'Image' and 'Image'